### Dengue Korean dataset

### Load dataset

In [2]:
# import
import pandas as pd
import numpy as np
import statistics
import scipy.stats as stats
import matplotlib.pyplot as plt
# import seaborn as sns
from scipy.stats import kurtosis
from scipy.stats import skew
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import mean_squared_error
#load the 2 csv file
# df1 = pd.read_csv('Singapore.csv')
# df2 = pd.read_csv('Weekly Dengue Cases.csv')
missing_value = ['N/a','na','nan',np.nan]
df1 = pd.read_csv('data/dengue_korea.csv', na_values=missing_value)

In [3]:
from IPython.display import display
isNull = df1.isnull().sum()
print(isNull)
display(df1)

_id                 0
disease             0
city or province    0
year                0
week                0
count               0
dtype: int64


,_id,disease,city or province,year,week,count
0,1,Dengue,Busan,2015,1,0
1,2,Dengue,Busan,2015,2,0
2,3,Dengue,Busan,2015,3,0
3,4,Dengue,Busan,2015,4,0
4,5,Dengue,Busan,2015,5,0
...,...,...,...,...,...,...
7101,7102,Dengue,Ulsan,2022,49,1
7102,7103,Dengue,Ulsan,2022,50,0
7103,7104,Dengue,Ulsan,2022,51,0
7104,7105,Dengue,Ulsan,2022,52,0


### Statistical Analysis
This section will be focused on identifying key series of the dataset and provide statistical summary of the data. The central tendency of the data can be used to describe the data by identifying summary statistics such as mean,median and mode.

In [4]:
# Measures Mean(average) of per year ############################
for i in range(2015,2022):
    print(i)
    filtered_df = df1.loc[(df1['year'] == i)]
    Dengue_mean = statistics.mean(filtered_df['count'])
    
    data = {'year': [i] ,
             'Avergae Dengue Cases': Dengue_mean}
    df = pd.DataFrame(data)
    display(df)

2015


,year,Avergae Dengue Cases
0,2015,0.288462


2016


,year,Avergae Dengue Cases
0,2016,0.348502


2017


...

,year,Avergae Dengue Cases
0,2019,0.308824


2020


,year,Avergae Dengue Cases
0,2020,0.049774


2021


,year,Avergae Dengue Cases
0,2021,0.003394


In [ ]:
# Measures Mode stat per year ############################
## Mode is the highest occurring value throughout the year

for i in range(2015,2022):
    filtered_df = df1.loc[(df1['year'] == i)]
    Dengue_mode = statistics.mode(filtered_df['count'])
    
    data = {'year': [i] ,
             'Mode Dengue Cases': Dengue_mode}
    df = pd.DataFrame(data)
    display(df)

,year,Mode Dengue Cases
0,2015,0.003394


,year,Mode Dengue Cases
0,2016,0.003394


,year,Mode Dengue Cases
0,2017,0.003394


,year,Mode Dengue Cases
0,2018,0.003394


,year,Mode Dengue Cases
0,2019,0.003394


,year,Mode Dengue Cases
0,2020,0.003394


,year,Mode Dengue Cases
0,2021,0.003394


In [6]:
df1.describe()

,_id,year,week,count
count,7106.000000,7106.000000,7106.000000,7106.000000
mean,3553.500000,2018.502392,26.626794,0.185899
std,2051.469839,2.295621,15.084702,0.612723
min,1.000000,2015.000000,1.000000,0.000000
25%,1777.250000,2016.000000,14.000000,0.000000
50%,3553.500000,2018.500000,27.000000,0.000000
75%,5329.750000,2021.000000,40.000000,0.000000
max,7106.000000,2022.000000,53.000000,8.000000


Because the describe function summaries the complete dataset rather than calculating on a per year basis, the per year analysis of the independent variables is significantly more comprehensive and beneficial if the user just wants to analyze how the data performed on selective years.

### Model Training

In [7]:
import pandas as pd
import io
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Load the data
df2 = pd.read_csv('data/dengue_korea.csv')

# df = pd.read_csv(io.StringIO(data))
df2 = df2.rename(columns={
    '_id': 'id',
    'disease': 'disease',
    'city or province': 'city_or_province',
    'year': 'year',
    'week': 'week',
    'count': 'count'
})
# display(df2['city_or_province'].unique())
cityProvinces = ['Busan', 'chungbuk', 'Chungnam', 'Daegu', 'Daejeon', 'Gangwon',
       'Gwangju', 'Gyeongbuk', 'Gyeongnam', 'Incheon', 'Jeju', 'Jeonbuk',
       'Jeonnam', 'Kyeong-gi', 'Sejong', 'Seoul', 'Ulsan']

# Map city names to their index in the list
city_to_index = {city: idx for idx, city in enumerate(cityProvinces)}
df2['city_index'] = df2['city_or_province'].map(city_to_index)

# drop unused columns
df2 = df2.drop(columns=['id', 'disease', 'city_or_province'])   

display(df2)



,year,week,count,city_index
0,2015,1,0,0
1,2015,2,0,0
2,2015,3,0,0
3,2015,4,0,0
4,2015,5,0,0
...,...,...,...,...
7101,2022,49,1,16
7102,2022,50,0,16
7103,2022,51,0,16
7104,2022,52,0,16


#### Linear Regression

In [8]:
# --- Feature Engineering ---
# Create a 'lag_1' feature: the count from the previous week
df2['lag_1'] = df2['count'].shift(1)

# Drop the first row since it has no previous week data (NaN)
df2 = df2.dropna()
display(df2)
# --- Model Training ---
# Define features (X) and target (y)
# We use the count from the previous week ('lag_1') to predict the current week's count ('count')
X = df2[['lag_1']]
y = df2['count']
# print(X)
# Split data - For time series, we usually don't shuffle
# Given the very small dataset, we'll use most of it for training
# Let's use the last 5 weeks for testing (a small test set)
split_point = len(df) - 5
X_train, X_test = X[:split_point], X[split_point:]
y_train, y_test = y[:split_point], y[split_point:]

# Initialize and train the model
model = LinearRegression()
model.fit(X_train, y_train)

,year,week,count,city_index,lag_1
1,2015,2,0,0,0.0
2,2015,3,0,0,0.0
3,2015,4,0,0,0.0
4,2015,5,0,0,0.0
5,2015,6,0,0,0.0
...,...,...,...,...,...
7101,2022,49,1,16,0.0
7102,2022,50,0,16,1.0
7103,2022,51,0,16,0.0
7104,2022,52,0,16,0.0


LinearRegression()

In [10]:
from sklearn.metrics import root_mean_squared_error
# --- Evaluation (on the small test set) ---
y_pred_test = model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred_test)
print(f"RMSE on test set: {rmse:.4f}")
print("\nTest Set Actual vs Predicted:")
test_results = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred_test})
print(test_results)


# --- Prediction ---
# Predict the count for the next week (week 22)
# We need the count from the last known week (week 21) as input
last_known_count = df2['count'].iloc[-1]
next_week_features = pd.DataFrame({'lag_1': [last_known_count]}) # Create a DataFrame for prediction input

prediction_next_week = model.predict(next_week_features)

# Since counts must be non-negative integers, we'll adjust the prediction
predicted_count = max(0, round(prediction_next_week[0]))

print(f"\nLast known count (Week 21): {last_known_count}")
print(f"Predicted count for Week 22: {predicted_count}")

# Display model coefficients (for understanding the simple linear model)
print(f"\nModel Coefficient (for lag_1): {model.coef_[0]:.4f}")
print(f"Model Intercept: {model.intercept_:.4f}")

RMSE on test set: 0.2784

Test Set Actual vs Predicted:
      Actual  Predicted
7102       0   0.523822
7103       0   0.108901
7104       0   0.108901
7105       0   0.108901

Last known count (Week 21): 0
Predicted count for Week 22: 0

Model Coefficient (for lag_1): 0.4149
Model Intercept: 0.1089


#### Other Two models
2. Random Forest
3. Gradient Boosting

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

features = ['year', 'week', 'city_index', 'lag_1']

X = df2[features]
y = df2['count']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    # 'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"{name}")
    print(f"  MSE: {mean_squared_error(y_test, y_pred):.2f}")
    print(f"  R2: {r2_score(y_test, y_pred):.2f}")


Random Forest
  MSE: 0.27
  R2: 0.28
Gradient Boosting
  MSE: 0.25
  R2: 0.35


#### LSTM

In [13]:
# from tensorflow.python.keras.models import Sequential 
# from tensorflow.python.keras.layers import Dense, LSTM
from keras.models import Sequential
from keras.layers import Dense, LSTM
from sklearn.preprocessing import MinMaxScaler

# model = Sequential()
# model.add(LSTM(256,input_shape=(X_train.shape[1],1)))
# model.add(Dense(1))
X = df2[['year', 'week', 'city_index', 'lag_1']]
y = df2['count']

# Normalize features
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1))

# Reshape for LSTM: [samples, time_steps, features]
# We'll use single timestep per input (time_steps=1)
X_lstm = X_scaled.reshape((X_scaled.shape[0], 1, X_scaled.shape[1]))

# Train-test split
X_train_lstm, X_test_lstm, y_train_lstm, y_test_lstm = train_test_split(
    X_lstm, y_scaled, test_size=0.2, random_state=42
)

print(f"X_train_lstm shape: {X_train_lstm.shape}")
print(f"X_test_lstm shape: {X_test_lstm.shape}")
print(f"y_train_lstm shape: {y_train_lstm.shape}")
print(f"y_test_lstm shape: {y_test_lstm.shape}")

# Build model
model = Sequential([
    LSTM(64, input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2]), return_sequences=False),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.fit(X_train_lstm, y_train_lstm, epochs=20, batch_size=32, verbose=1)

# Predict
y_pred_lstm = model.predict(X_test_lstm)
y_pred_lstm = scaler_y.inverse_transform(y_pred_lstm)
y_test_lstm = scaler_y.inverse_transform(y_test_lstm)

# Evaluate
print("LSTM:")
print(f"  MSE: {mean_squared_error(y_test_lstm, y_pred_lstm):.2f}")
print(f"  R2: {r2_score(y_test_lstm, y_pred_lstm):.2f}")

X_train_lstm shape: (5684, 1, 4)
X_test_lstm shape: (1421, 1, 4)
y_train_lstm shape: (5684, 1)
y_test_lstm shape: (1421, 1)


/Users/rattanak/miniconda3/envs/py312/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
178/178 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.0053
Epoch 2/20
178/178 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0048
Epoch 3/20
178/178 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0046
Epoch 4/20
178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0049
Epoch 5/20
178/178 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0049
Epoch 6/20
178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0052
Epoch 7/20
178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0046
Epoch 8/20
178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0044
Epoch 9/20
178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0052
Epoch 10/20
178/178 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0045
Epoch 11/20
178/178 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0042
Epoch 12/20
178/178 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0053
Epoch 13/20
178/178 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0049
Epoch 14/20
178/178 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0046
Epoch 15/20
178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - lo

### Predict the output using new data

In [ ]:
# Predict the output using new data
import pandas as pd

# Simulate new input data
new_data = pd.DataFrame({
    'year': [2023]*4,
    'week': [1, 2, 3, 4],
    'city_index': [0]*4,      # Busan
    'lag_1': [0.0, 1.0, 2.0, 1.0]  # Hypothetical recent counts
})

display(new_data)
new_data1 = new_data.copy()
preds = models['Random Forest'].predict(new_data1[features])
new_data1['predicted_count'] = preds
print("\n==============New Data with Predictions==============")
print("Random Forest Predictions")
print(new_data1)


# Example: Predict using trained Gradient Boosting model
# Simulate new input data
new_data2 = new_data.copy()
preds = models['Gradient Boosting'].predict(new_data)
new_data2['predicted_count'] = preds
print("\n==============New Data with Predictions==============")
print("Gradient Boosting Predictions")
print(new_data2)

# Example: Predict using trained LSTM model
new_data3 = new_data.copy()
new_data3_scaled = scaler_X.transform(new_data3[features])
new_data3_lstm = new_data3_scaled.reshape((new_data3_scaled.shape[0], 1, new_data3_scaled.shape[1]))
y_pred_lstm_new = model.predict(new_data3_lstm)
y_pred_lstm_new = scaler_y.inverse_transform(y_pred_lstm_new)
new_data3['predicted_count'] = y_pred_lstm_new
print("\n==============New Data with Predictions==============")
print("LSTM Predictions")
print(new_data3)

,year,week,city_index,lag_1
0,2023,1,0,0.0
1,2023,2,0,1.0
2,2023,3,0,2.0
3,2023,4,0,1.0


==============New Data with Predictions==============
Random Forest Predictions
   year  week  city_index  lag_1  predicted_count
0  2023     1           0    0.0             0.00
1  2023     2           0    1.0             0.54
2  2023     3           0    2.0             0.92
3  2023     4           0    1.0             0.21
==============New Data with Predictions==============
Gradient Boosting Predictions
   year  week  city_index  lag_1  predicted_count
0  2023     1           0    0.0         0.064637
1  2023     2           0    1.0         0.115978
2  2023     3           0    2.0         0.135872
3  2023     4           0    1.0         0.115978
